# 02 — Data Cleaning and Metric Computation

This notebook takes the raw shot log from `data/raw/shots_2025_26.csv` and prepares it for all downstream analysis. It is the single data pipeline step between the raw API pull and every analysis notebook that follows.

What this notebook does, in order:

1. Load the raw 2025-26 NBA shot log
2. Clean and convert shot coordinates from tenths of feet to feet with the basket at the origin
3. Filter out rows with missing or physically impossible coordinates
4. Assign each shot to one of 11 named court zones using geometry from `src/metrics.py`, with SHOT_TYPE as the final arbiter at zone boundaries — no shot can appear in both a 2-point and a 3-point zone
5. Apply the 300-shot minimum filter at the player level — players below this threshold are excluded from the entire dataset
6. Compute Points Per Shot (PPS) and shot frequency per player per zone with no zone-level minimum filter — every zone with at least 1 attempt is included
7. Compute league average PPS per zone across all qualifying players
8. Save the enriched shot-level data to `data/processed/shots_cleaned.csv`
9. Save the zone-level summary table to `data/processed/zone_stats_2025_26.csv`

The only efficiency metric used in this notebook and throughout the project is PPS. A made 2-pointer is worth 2 points, a made 3-pointer is worth 3 points, and any miss is worth 0 points. eFG% does not appear anywhere.

## Section 1.1 — Imports, Constants, and Paths

All imports for this notebook live here in a single cell. Nothing is imported anywhere else. Running this cell first is the only setup step required before any other cell.

Constants defined here:

`SEASON` — the NBA season this analysis covers. Changing this one value updates all file paths automatically.

`MIN_PLAYER_FGA` — the minimum number of total shot attempts required for a player to be included in the analysis. Set to 300 per the project definition.

`ZONE_ORDER` — the canonical ordering of all 11 court zones used in every table and chart throughout the project. Defined once here and referenced everywhere else.

`THREE_PT_ZONES` — the set of zone names that are 3-point territory. Used in the SHOT_TYPE arbiter step to resolve coordinate/classification conflicts at zone boundaries.

File paths are all built programmatically from `PROJECT_ROOT` so they work correctly regardless of where on your machine the repo lives.

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Add the project root to Python's module search path so we can import from src/
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.metrics import assign_zone   # geometry-based zone classifier — never redefine this here

# ── Season and player pool ─────────────────────────────────────────────────────
SEASON         = "2025-26"   # change this to re-run for a different season
MIN_PLAYER_FGA = 300         # players below this total shot count are excluded

# ── Zone ordering — defined once, used throughout ─────────────────────────────
ZONE_ORDER = [
    "Restricted Area",
    "Paint (Non-RA)",
    "Mid-Range Left",
    "Mid-Range Center",
    "Mid-Range Right",
    "Left Corner 3",
    "Right Corner 3",
    "Above Break 3 Left",
    "Above Break 3 Center",
    "Above Break 3 Right",
    "Backcourt",
]

# ── 3-point zones — used by the SHOT_TYPE arbiter in Section 2.1 ──────────────
THREE_PT_ZONES = frozenset({
    "Left Corner 3",
    "Right Corner 3",
    "Above Break 3 Left",
    "Above Break 3 Center",
    "Above Break 3 Right",
})

# ── File paths — all derived from PROJECT_ROOT ────────────────────────────────
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

_season_slug   = SEASON.replace("-", "_")           # "2025-26" → "2025_26"
RAW_FILE       = RAW_DIR       / f"shots_{_season_slug}.csv"
SHOTS_OUT      = PROCESSED_DIR / "shots_cleaned.csv"
ZONE_STATS_OUT = PROCESSED_DIR / f"zone_stats_{_season_slug}.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ZONE_STATS_OUT = PROCESSED_DIR / f"zone_stats_{_season_slug}.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ── Confirmation print ────────────────────────────────────────────────────────
print(f"Season          : {SEASON}")
print(f"Min player FGA  : {MIN_PLAYER_FGA}")
print(f"Raw input       : {RAW_FILE}")
print(f"Shots output    : {SHOTS_OUT}")
print(f"Zone stats out  : {ZONE_STATS_OUT}")
print(f"assign_zone     : imported from src.metrics")
print(f"Zones defined   : {len(ZONE_ORDER)}")

Season          : 2025-26
Min player FGA  : 300
Raw input       : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/raw/shots_2025_26.csv
Shots output    : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/processed/shots_cleaned.csv
Zone stats out  : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/processed/zone_stats_2025_26.csv
assign_zone     : imported from src.metrics
Zones defined   : 11


## Section 1.2 — Load the Raw Data

We read the CSV saved by Notebook 01 into a DataFrame called `raw` and leave it completely untouched. Every cleaning step that follows works on a copy so we can always compare the cleaned result against the original if something looks wrong.

Three checks tell us what we are starting with:

`.shape` confirms the row count (shot attempts) and column count. Based on the Notebook 01 pull we expect roughly 219,000 rows.

`.head(3)` gives a visual confirmation that column names and values look reasonable before any code runs on them.

`.info()` shows every column's data type and its non-null count. A column with fewer non-null entries than total rows has missing values that need handling. We want to see zero nulls across all columns at this stage.

In [2]:
raw = pd.read_csv(RAW_FILE)

print(f"Shape: {raw.shape[0]:,} rows  x  {raw.shape[1]} columns\n")

print("First 3 rows:")
display(raw.head(3))

print("\nColumn types and null counts:")
raw.info()

Shape: 219,160 rows  x  25 columns

First 3 rows:


,GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,...,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,NAME
0,Shot Chart Detail,22500038,224,1630173,Precious Achiuwa,1610612758,Sacramento Kings,2,7,54,...,24+ ft.,27,-50,274,1,0,20251107,SAC,OKC,Precious Achiuwa
1,Shot Chart Detail,22500038,259,1630173,Precious Achiuwa,1610612758,Sacramento Kings,2,5,27,...,Less Than 8 ft.,4,-40,8,1,1,20251107,SAC,OKC,Precious Achiuwa
2,Shot Chart Detail,22500038,659,1630173,Precious Achiuwa,1610612758,Sacramento Kings,4,3,48,...,Less Than 8 ft.,4,17,40,1,0,20251107,SAC,OKC,Precious Achiuwa



Column types and null counts:
<class 'pandas.DataFrame'>
RangeIndex: 219160 entries, 0 to 219159
Data columns (total 25 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   GRID_TYPE            219160 non-null  str  
 1   GAME_ID              219160 non-null  int64
 2   GAME_EVENT_ID        219160 non-null  int64
 3   PLAYER_ID            219160 non-null  int64
 4   PLAYER_NAME          219160 non-null  str  
 5   TEAM_ID              219160 non-null  int64
 6   TEAM_NAME            219160 non-null  str  
 7   PERIOD               219160 non-null  int64
 8   MINUTES_REMAINING    219160 non-null  int64
 9   SECONDS_REMAINING    219160 non-null  int64
 10  EVENT_TYPE           219160 non-null  str  
 11  ACTION_TYPE          219160 non-null  str  
 12  SHOT_TYPE            219160 non-null  str  
 13  SHOT_ZONE_BASIC      219160 non-null  str  
 14  SHOT_ZONE_AREA       219160 non-null  str  
 15  SHOT_ZONE_RANGE      219160 non

## Section 1.3 — Understanding the NBA API Coordinate System

Before cleaning the coordinates, we need to understand exactly what the raw values mean. The NBA API encodes shot locations in a non-obvious way and getting this wrong corrupts every zone assignment that follows.

**Unit:** LOC_X and LOC_Y are in tenths of a foot, not feet. A raw value of 237 means 23.7 feet, not

**Origin:** The basket sits at (0, 0). All distances are measured from the center of the rim.

**LOC_X — horizontal position:**
Negative values are the left side of the court, positive are the right side. The sidelines sit at roughly ±250 in raw units (±25 feet). A left corner three will show LOC_X near −220 (−22 feet).

**LOC_Y — depth from the basket toward half court:**
Zero is directly at the basket. Positive values go toward half court. The baseline behind the basket sits at around −47 in raw units (−4.75 feet). A shot at the top of the three-point arc will show LOC_Y near 237 (23.7 feet).

The code cell below prints descriptive statistics for both columns so we can see the actual range of values in this season's data and spot anything unexpected before proceeding. We also save the raw min and max values here so we can print a before/after comparison after coordinate cleaning is complete.

In [3]:
# Confirm no nulls — we expect zero from Section 1.2 but verify explicitly
# because the cleaning step in Section 1.4 depends on this being true
null_x = raw["LOC_X"].isna().sum()
null_y = raw["LOC_Y"].isna().sum()
print(f"Null values — LOC_X: {null_x}  |  LOC_Y: {null_y}\n")

# Descriptive statistics for both coordinate columns in raw tenths-of-foot units
print("Raw coordinate statistics (tenths of a foot):\n")
print(raw[["LOC_X", "LOC_Y"]].describe().round(1).to_string())

# Save raw ranges now — used for the before/after summary in Section 1.6
RAW_X_RANGE = (int(raw["LOC_X"].min()), int(raw["LOC_X"].max()))
RAW_Y_RANGE = (int(raw["LOC_Y"].min()), int(raw["LOC_Y"].max()))

print(f"\nLOC_X range : {RAW_X_RANGE[0]} → {RAW_X_RANGE[1]}  "
      f"(expected roughly −250 → 250)")
print(f"LOC_Y range : {RAW_Y_RANGE[0]} → {RAW_Y_RANGE[1]}  "
      f"(expected roughly −50 → 750)")

Null values — LOC_X: 0  |  LOC_Y: 0

Raw coordinate statistics (tenths of a foot):

          LOC_X     LOC_Y
count  219160.0  219160.0
mean       -2.8      95.6
std       116.4      93.5
min      -250.0     -51.0
25%       -53.0      14.0
50%         0.0      57.0
75%        45.0     183.0
max       250.0     734.0

LOC_X range : -250 → 250  (expected roughly −250 → 250)
LOC_Y range : -51 → 734  (expected roughly −50 → 750)


## Section 1.4 — Filter Out Invalid Coordinates

The NBA API data is generally clean, but a small number of rows can carry coordinate values that fall outside the physical dimensions of an NBA court. These usually represent encoding artifacts where a missing location was stored as a default value rather than left null.

We create a working copy called `cleaned` here and apply all remaining transformations to it. The original `raw` DataFrame stays untouched throughout the notebook so we always have something to compare against.

The bounds we use for filtering:

LOC_X must be between −260 and 260 raw units (−26 to 26 feet). The actual sidelines are at ±25 feet (±250 raw units). We use a slightly wider window so we do not accidentally clip any legitimate corner three attempts that the API may have logged with a coordinate just beyond the sideline.

LOC_Y must be between −80 and 900 raw units (−8 to 90 feet). The baseline behind the basket sits at roughly −4.75 feet (−47 raw units). We use −80 to give a small buffer. The upper bound of 900 (90 feet) covers everything from normal shots up to full-court heaves — any shot the API recorded as an attempt is kept here. The zone assignment step later will assign anything beyond half court (47 feet) to the Backcourt zone.

We print a summary showing how many rows were removed so we can confirm the filter is not too aggressive.

In [4]:
# Create the working copy — raw stays untouched from this point forward
cleaned = raw.copy()

n_before = len(cleaned)

# Filter: keep only rows within the physical bounds of an NBA court
coord_mask = (
    (cleaned["LOC_X"] >= -260) & (cleaned["LOC_X"] <= 260) &
    (cleaned["LOC_Y"] >= -80)  & (cleaned["LOC_Y"] <= 900)
)
cleaned = cleaned[coord_mask].copy()

n_after   = len(cleaned)
n_dropped = n_before - n_after

print(f"Rows before filter : {n_before:,}")
print(f"Rows after filter  : {n_after:,}")
print(f"Rows dropped       : {n_dropped:,}  ({n_dropped / n_before:.3%} of total)")

if n_dropped > 0:
    print("\nDropped row coordinate ranges:")
    dropped = raw[~coord_mask]
    print(f"  LOC_X : {dropped['LOC_X'].min()} → {dropped['LOC_X'].max()}")
    print(f"  LOC_Y : {dropped['LOC_Y'].min()} → {dropped['LOC_Y'].max()}")

Rows before filter : 219,160
Rows after filter  : 219,160
Rows dropped       : 0  (0.000% of total)


## Section 1.5 — Convert Coordinates to Feet

We divide LOC_X and LOC_Y by 10 to convert from the API's tenths-of-a-foot encoding to feet. The result goes into two new columns — LOC_X_FT and LOC_Y_FT — rather than overwriting the originals. Keeping both lets us cross-check if anything looks off later.

After conversion, known shot locations should land at predictable coordinates:

A left corner three should show LOC_X_FT near −22, LOC_Y_FT near 0 to 5.

A shot at the free-throw line should show LOC_X_FT near 0, LOC_Y_FT near 14 to 15.

A top-of-the-key three should show straight-line distance from (0, 0) of approximately 23.75 feet.

A dunk or layup in the restricted area should show distance under 4 feet.

The spot-check at the bottom of the code cell samples five rows from the data and shows the raw and converted values side by side so we can visually confirm the conversion is correct before moving on.

In [5]:
# Divide by 10 to convert tenths-of-a-foot to feet
cleaned["LOC_X_FT"] = cleaned["LOC_X"] / 10.0
cleaned["LOC_Y_FT"] = cleaned["LOC_Y"] / 10.0

# Compute straight-line distance from the basket for spot-check verification
cleaned["DIST_FT"] = np.sqrt(cleaned["LOC_X_FT"]**2 + cleaned["LOC_Y_FT"]**2).round(2)

# Spot-check: sample one row from each of four expected coordinate ranges
# to confirm the conversion matches known NBA court geometry
checks = pd.concat([
    cleaned[cleaned["SHOT_ZONE_BASIC"] == "Restricted Area"].sample(1, random_state=1),
    cleaned[cleaned["SHOT_ZONE_BASIC"] == "In The Paint (Non-RA)"].sample(1, random_state=1),
    cleaned[cleaned["SHOT_ZONE_BASIC"] == "Left Corner 3"].sample(1, random_state=1),
    cleaned[cleaned["SHOT_ZONE_BASIC"] == "Above the Break 3"].sample(1, random_state=1),
])

spot_cols = ["NAME", "SHOT_ZONE_BASIC", "LOC_X", "LOC_Y", "LOC_X_FT", "LOC_Y_FT", "DIST_FT"]
print("Spot-check — one row per zone type:\n")
display(checks[spot_cols].reset_index(drop=True))

print(f"\nNew columns added to cleaned:")
print(f"  LOC_X_FT  range: {cleaned['LOC_X_FT'].min():.1f} → {cleaned['LOC_X_FT'].max():.1f} ft")
print(f"  LOC_Y_FT  range: {cleaned['LOC_Y_FT'].min():.1f} → {cleaned['LOC_Y_FT'].max():.1f} ft")
print(f"  DIST_FT   range: {cleaned['DIST_FT'].min():.1f} → {cleaned['DIST_FT'].max():.1f} ft")

Spot-check — one row per zone type:



,NAME,SHOT_ZONE_BASIC,LOC_X,LOC_Y,LOC_X_FT,LOC_Y_FT,DIST_FT
0,Tyrese Maxey,Restricted Area,10,-1,1.0,-0.1,1.00
1,Jaylen Brown,In The Paint (Non-RA),47,2,4.7,0.2,4.70
2,Donte DiVincenzo,Left Corner 3,-225,-5,-22.5,-0.5,22.51
3,Brandin Podziemski,Above the Break 3,-37,290,-3.7,29.0,29.24



New columns added to cleaned:
  LOC_X_FT  range: -25.0 → 25.0 ft
  LOC_Y_FT  range: -5.1 → 73.4 ft
  DIST_FT   range: 0.0 → 73.4 ft


## Section 1.6 — Before/After Summary

This is the acceptance check for the entire coordinate cleaning pipeline. We compare the raw ranges saved in Section 1.3 against the converted ranges in `cleaned` to confirm that the divide-by-10 conversion is exact, no rows were lost that should have been kept, and the cleaned values fall within expected NBA court dimensions.

If everything is correct, the cleaned foot values should be exactly one tenth of the raw tenths-of-a-foot values, the row count should match what we saw after the filter in Section 1.4, and the player count should match what Notebook 01 pulled.

This is the last cell in Section 1. Section 2 begins zone assignment.

In [6]:
print("=== Coordinate cleaning — before / after ===\n")

print(f"{'':30s}  {'RAW (tenths of ft)':>20s}  {'CLEANED (ft)':>15s}")
print(f"{'─'*70}")
print(f"{'LOC_X range':30s}  "
      f"{str(RAW_X_RANGE[0]) + ' → ' + str(RAW_X_RANGE[1]):>20s}  "
      f"{cleaned['LOC_X_FT'].min():.1f} → {cleaned['LOC_X_FT'].max():.1f}")
print(f"{'LOC_Y range':30s}  "
      f"{str(RAW_Y_RANGE[0]) + ' → ' + str(RAW_Y_RANGE[1]):>20s}  "
      f"{cleaned['LOC_Y_FT'].min():.1f} → {cleaned['LOC_Y_FT'].max():.1f}")

print(f"\n{'=== Dataset summary after Section 1 ==='}")
print(f"Total rows     : {len(cleaned):,}")
print(f"Unique players : {cleaned['NAME'].nunique():,}")
print(f"Columns        : {len(cleaned.columns)}  "
      f"(original {len(raw.columns)} + LOC_X_FT, LOC_Y_FT, DIST_FT)")

print(f"\nSection 1 complete. Proceeding to zone assignment.")

=== Coordinate cleaning — before / after ===

                                  RAW (tenths of ft)     CLEANED (ft)
──────────────────────────────────────────────────────────────────────
LOC_X range                               -250 → 250  -25.0 → 25.0
LOC_Y range                                -51 → 734  -5.1 → 73.4

=== Dataset summary after Section 1 ===
Total rows     : 219,160
Unique players : 582
Columns        : 28  (original 25 + LOC_X_FT, LOC_Y_FT, DIST_FT)

Section 1 complete. Proceeding to zone assignment.


## Section 2 — Zone Assignment and Base Metrics

Every shot in our dataset has an `(LOC_X_FT, LOC_Y_FT)` coordinate. Now we assign each one a named court zone, then compute shooting efficiency metrics per player per zone.

**A note  within ±26 ft, LOC_Y within −8 to 90 ft) are wider than any valid shot location, so no legitimate corner 3s or above-break 3s were removed. No changes needed.

**Section 2 steps:**
1. Define a zone assignment function using NBA court geometry
2. Apply it to every shot and verify no shots fall through as unassigned
3. Compute FG%, eFG%, PPS, and shot frequency per player per zone
4. Save the enriched DataFrame to `data/processed/shots_cleaned.csv`

## Section 2.1 — Zone Assignment

Every shot now has clean foot-unit coordinates. This section assigns each shot to one of 11 named court zones and adds two helper columns needed for PPS computation.

The three new columns added here:

`IS_3PT` — 1 if the shot was a 3-point attempt, 0 if 2-point. Derived directly from the SHOT_TYPE column that the NBA API provides. SHOT_TYPE is the authoritative source — coordinates alone cannot reliably distinguish a 2-point from a 3-point attempt at the exact boundary.

`POINTS` — how many points the shot produced. Made 3-pointer = 3, made 2-pointer = 2, any miss = 0. This is the numerator in every PPS calculation.

`ZONE` — one of the 11 named court zones. Assigned in two passes. Pass 1 uses the geometry function from src.metrics (pure coordinate math). Pass 2 runs the SHOT_TYPE arbiter, which corrects any row where the geometry and the official shot classification disagree at a zone boundary.

In [7]:
# IS_3PT: read directly from SHOT_TYPE — the API's own column is authoritative.
# "3PT Field Goal" starts with "3PT", everything else is a 2-point attempt.
cleaned["IS_3PT"] = cleaned["SHOT_TYPE"].str.startswith("3PT").astype(int)

# POINTS: made 3-pointer = 3, made 2-pointer = 2, any miss = 0.
# SHOT_MADE_FLAG is 1 if made and 0 if missed, so multiplying handles the miss case.
cleaned["POINTS"] = cleaned["SHOT_MADE_FLAG"] * (2 + cleaned["IS_3PT"])

# Quick verification: totals should be internally consistent
n_3pt_attempts = cleaned["IS_3PT"].sum()
n_2pt_attempts = (cleaned["IS_3PT"] == 0).sum()
total_points   = cleaned["POINTS"].sum()
print(f"3PT attempts : {n_3pt_attempts:,}")
print(f"2PT attempts : {n_2pt_attempts:,}")
print(f"Total points scored across all shots: {total_points:,}")
print(f"Implied points per shot (all zones, all players): "
      f"{total_points / len(cleaned):.3f}")

3PT attempts : 90,970
2PT attempts : 128,190
Total points scored across all shots: 239,171
Implied points per shot (all zones, all players): 1.091


### Zone assignment — geometry pass then SHOT_TYPE arbiter

Pass 1 calls `assign_zone()` from `src.metrics` for every row using LOC_X_FT and LOC_Y_FT. This function uses NBA court geometry to place each shot into one of the 11 zones. The apply() call across 219,000 rows takes roughly 10 to 15 seconds — the print statement will confirm when it finishes.

Pass 2 is the SHOT_TYPE arbiter. A small number of shots land exactly on a zone boundary where the coordinate-based geometry and the official SHOT_TYPE classification disagree. For these shots, SHOT_TYPE wins. Any row where IS_3PT is 0 but the geometry placed it in a 3-point zone gets pushed to the nearest mid-range zone. Any row where IS_3PT is 1 but the geometry placed it in a 2-point zone gets pushed to the nearest above-break-3 zone. After the arbiter runs, there are zero remaining conflicts.

In [8]:
# ── Pass 1: geometry-based zone assignment ────────────────────────────────────
# assign_zone is imported from src.metrics — it is never redefined in this notebook.
print("Assigning zones via src.metrics.assign_zone() — takes ~10-15 seconds ...")
cleaned["ZONE"] = cleaned.apply(
    lambda r: assign_zone(r["LOC_X_FT"], r["LOC_Y_FT"]),
    axis=1
)
print("Geometry pass complete.\n")

# ── Pass 2: SHOT_TYPE arbiter ─────────────────────────────────────────────────
# THREE_PT_ZONES is defined in Section 1.1 — the frozenset of 3-point zone names.
in_3pt_zone = cleaned["ZONE"].isin(THREE_PT_ZONES)
x = cleaned["LOC_X_FT"].values   # numpy array for vectorized comparison

# Case A: IS_3PT == 0 but geometry said 3PT zone — override to mid-range
conflict_2pt = (cleaned["IS_3PT"] == 0) & in_3pt_zone

# Case B: IS_3PT == 1 but geometry said 2PT zone — override to above-break 3
# Backcourt stays as Backcourt even if SHOT_TYPE says 3PT — it is not a valid
# shooting zone and should not be reclassified as above-break 3.
conflict_3pt = (
    (cleaned["IS_3PT"] == 1) &
    ~in_3pt_zone &
    (cleaned["ZONE"] != "Backcourt")
)

# Build full-length correction arrays then select only the conflicting rows
mid_zone = np.where(x < -7.0, "Mid-Range Left",
           np.where(x <= 7.0, "Mid-Range Center", "Mid-Range Right"))

ab3_zone = np.where(x < -7.0, "Above Break 3 Left",
           np.where(x <= 7.0, "Above Break 3 Center", "Above Break 3 Right"))

cleaned.loc[conflict_2pt, "ZONE"] = mid_zone[conflict_2pt.values]
cleaned.loc[conflict_3pt, "ZONE"] = ab3_zone[conflict_3pt.values]

# ── Verification ──────────────────────────────────────────────────────────────
n_null = cleaned["ZONE"].isna().sum()
n_remaining = (
    ((cleaned["IS_3PT"] == 0) & cleaned["ZONE"].isin(THREE_PT_ZONES)) |
    ((cleaned["IS_3PT"] == 1) & ~cleaned["ZONE"].isin(THREE_PT_ZONES) & (cleaned["ZONE"] != "Backcourt"))
).sum()

print(f"Unassigned shots          : {n_null}  (must be 0)")
print(f"Arbiter — 2PT in 3PT zone : {conflict_2pt.sum()} corrected")
print(f"Arbiter — 3PT in 2PT zone : {conflict_3pt.sum()} corrected")
print(f"Remaining conflicts       : {n_remaining}  (must be 0)")

# ── Zone distribution ─────────────────────────────────────────────────────────
total = len(cleaned)
zone_counts = cleaned["ZONE"].value_counts().reindex(ZONE_ORDER, fill_value=0)

print(f"\nZone distribution — {cleaned['NAME'].nunique()} players, {total:,} shots:\n")
print(f"  {'Zone':<28} {'Shots':>8}   {'% Total':>8}")
print(f"  {'─' * 48}")
for zone, count in zone_counts.items():
    print(f"  {zone:<28} {count:>8,}   {count / total * 100:>7.1f}%")
print(f"  {'─' * 48}")
print(f"  {'Total':<28} {total:>8,}   {'100.0%':>8}")

Assigning zones via src.metrics.assign_zone() — takes ~10-15 seconds ...
Geometry pass complete.

Unassigned shots          : 0  (must be 0)
Arbiter — 2PT in 3PT zone : 12 corrected
Arbiter — 3PT in 2PT zone : 8 corrected
Remaining conflicts       : 0  (must be 0)

Zone distribution — 582 players, 219,160 shots:

  Zone                            Shots    % Total
  ────────────────────────────────────────────────
  Restricted Area                62,278      28.4%
  Paint (Non-RA)                 44,573      20.3%
  Mid-Range Left                  8,312       3.8%
  Mid-Range Center                4,748       2.2%
  Mid-Range Right                 8,279       3.8%
  Left Corner 3                  12,308       5.6%
  Right Corner 3                 11,460       5.2%
  Above Break 3 Left             27,637      12.6%
  Above Break 3 Center           14,827       6.8%
  Above Break 3 Right            24,723      11.3%
  Backcourt                          15       0.0%
  ────────────────────

## Section 2.2 — Apply the 300-Shot Player-Level Filter

The project defines its player pool as all active 2025-26 players with at least 300 total shot attempts. Players below this threshold are excluded from the entire dataset — their shots do not appear in shots_cleaned.csv and they do not contribute to any PPS calculation, zone average, or league baseline downstream.

The threshold is applied once here and carried through every notebook that follows. The constant MIN_PLAYER_FGA defined in Section 1.1 controls this value so it never needs to be changed in more than one place.

The filter operates at the player level only. There is no zone-level minimum. A qualifying player who took only 1 shot from a particular zone still has that zone included in their profile — a low-attempt zone is an analytical signal in itself and should not be silently dropped.

After this cell runs, `cleaned` refers only to shots from qualifying players. Every cell from this point forward works on the filtered dataset.

In [9]:
# Count total shot attempts per player — every row in cleaned is one attempt
player_totals = cleaned.groupby("NAME").size().rename("TOTAL_FGA").sort_values(ascending=False)

n_before    = player_totals.shape[0]
qualifying  = player_totals[player_totals >= MIN_PLAYER_FGA]
excluded    = player_totals[player_totals <  MIN_PLAYER_FGA]

# Replace cleaned with only the qualifying players' shots
# .copy() ensures the filtered DataFrame is independent — no SettingWithCopyWarning downstream
cleaned = cleaned[cleaned["NAME"].isin(qualifying.index)].copy()

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Player-level filter  (MIN_PLAYER_FGA = {MIN_PLAYER_FGA})\n")
print(f"  Players before filter  : {n_before:,}")
print(f"  Players qualified      : {len(qualifying):,}")
print(f"  Players excluded       : {len(excluded):,}")

print(f"\nQualifying players — attempt range:")
print(f"  Fewest attempts  : {qualifying.min():,}  ({qualifying.idxmin()})")
print(f"  Most attempts    : {qualifying.max():,}  ({qualifying.idxmax()})")
print(f"  Median attempts  : {int(qualifying.median()):,}")

print(f"\nExcluded players — attempt range:")
print(f"  Fewest attempts  : {excluded.min():,}  ({excluded.idxmin()})")
print(f"  Most attempts    : {excluded.max():,}  ({excluded.idxmax()})")

print(f"\nFiltered dataset:")
print(f"  Rows             : {len(cleaned):,}")
print(f"  Unique players   : {cleaned['NAME'].nunique():,}")
print(f"  Rows removed     : {219_160 - len(cleaned):,}")

Player-level filter  (MIN_PLAYER_FGA = 300)

  Players before filter  : 582
  Players qualified      : 284
  Players excluded       : 298

Qualifying players — attempt range:
  Fewest attempts  : 300  (Keegan Murray)
  Most attempts    : 1,543  (Jaylen Brown)
  Median attempts  : 587

Excluded players — attempt range:
  Fewest attempts  : 1  (Tosan Evbuomwan)
  Most attempts    : 296  (Jamir Watkins)

Filtered dataset:
  Rows             : 185,684
  Unique players   : 284
  Rows removed     : 33,476


## Section 2.3 — Compute PPS and Shot Frequency Per Player Per Zone

Now that the dataset contains only the 284 qualifying players, we aggregate to the player-zone level. This is the core computation the entire project is built around.

For each player-zone combination we compute:

`FGA` — total shot attempts from that zone. Every row in the shot-level data is one attempt, so this is simply the row count for that player-zone group.

`POINTS` — total points produced from that zone. Summed from the POINTS column added in Section 2.1.

`PPS` — Points Per Shot. Total points divided by total attempts. This is the only efficiency metric used in this project. A player who makes 40 percent of their corner threes generates 0.40 × 3 = 1.20 PPS from that zone. A player who makes 55 percent of their restricted area attempts generates 0.55 × 2 = 1.10 PPS.

`SHOT_FREQ` — the fraction of the player's total attempts that came from this zone, stored as a decimal. A value of 0.35 means 35 percent of the player's shots came from that zone. Shot frequencies across all zones sum to exactly 1.0 for every player.

No zone-level minimum filter is applied. If a player took one shot from a zone, that zone appears in their profile with FGA = 1. A single attempt in a zone is itself an analytical signal and is never excluded.

The verification table at the bottom shows Stephen Curry's full zone profile so we can confirm the numbers are internally consistent before saving anything.

In [10]:
# ── Aggregate to player-zone level ───────────────────────────────────────────
zone_metrics = (
    cleaned
    .groupby(["NAME", "ZONE"])
    .agg(
        FGA    = ("SHOT_MADE_FLAG", "count"),  # row count = total attempts (made + missed)
        POINTS = ("POINTS", "sum"),            # total points produced from this zone
    )
    .reset_index()
)

# PPS: total points / total attempts for this player-zone combination
zone_metrics["PPS"] = (zone_metrics["POINTS"] / zone_metrics["FGA"]).round(4)

# SHOT_FREQ: this zone's attempts as a fraction of the player's total attempts
# transform("sum") broadcasts each player's total back to all their zone rows
player_total_fga = zone_metrics.groupby("NAME")["FGA"].transform("sum")
zone_metrics["SHOT_FREQ"] = (zone_metrics["FGA"] / player_total_fga).round(4)

# ── Shape summary ─────────────────────────────────────────────────────────────
n_rows    = len(zone_metrics)
n_players = zone_metrics["NAME"].nunique()
print(f"zone_metrics shape   : {n_rows:,} rows x {zone_metrics.shape[1]} columns")
print(f"Qualifying players   : {n_players:,}")
print(f"Avg zones per player : {n_rows / n_players:.1f}")

# ── Verification: Stephen Curry ───────────────────────────────────────────────
curry = zone_metrics[zone_metrics["NAME"] == "Stephen Curry"].copy()

curry_total_fga  = curry["FGA"].sum()
curry_total_pts  = curry["POINTS"].sum()
curry_freq_sum   = curry["SHOT_FREQ"].sum()

print(f"\nStephen Curry — {curry_total_fga:,} total attempts, "
      f"{curry_total_pts:,} total points, "
      f"overall PPS = {curry_total_pts / curry_total_fga:.4f}")
print(f"SHOT_FREQ sum across all zones : {curry_freq_sum:.4f}  (must be 1.0000)\n")

# Display table: reindex to ZONE_ORDER so zones appear in standard order
# Zones where Curry has zero attempts show as NaN — correctly absent from his profile
curry_display = (
    curry.set_index("ZONE")
    .reindex(ZONE_ORDER)
    [["FGA", "POINTS", "PPS", "SHOT_FREQ"]]
    .copy()
)
curry_display["SHOT_FREQ"] = (curry_display["SHOT_FREQ"] * 100).round(1)
curry_display.columns      = ["FGA", "POINTS", "PPS", "SHOT_FREQ (%)"]

display(curry_display)


zone_metrics shape   : 2,778 rows x 6 columns
Qualifying players   : 284
Avg zones per player : 9.8

Stephen Curry — 799 total attempts, 938 total points, overall PPS = 1.1740
SHOT_FREQ sum across all zones : 1.0000  (must be 1.0000)



,FGA,POINTS,PPS,SHOT_FREQ (%)
ZONE,,,,
Restricted Area,122.0,168.0,1.3770,15.3
Paint (Non-RA),101.0,104.0,1.0297,12.6
Mid-Range Left,41.0,40.0,0.9756,5.1
Mid-Range Center,15.0,18.0,1.2000,1.9
Mid-Range Right,36.0,38.0,1.0556,4.5
Left Corner 3,20.0,33.0,1.6500,2.5
Right Corner 3,13.0,21.0,1.6154,1.6
Above Break 3 Left,152.0,180.0,1.1842,19.0
Above Break 3 Center,121.0,144.0,1.1901,15.1


## Section 2.4 — League Average PPS Per Zone

The league average PPS per zone is the comparison baseline used throughout every downstream notebook. For each zone, it answers the question: across all 284 qualifying players combined, how many points does a shot from this zone produce on average?

The correct formula is total points generated in the zone across all qualifying players divided by total shot attempts in the zone across all qualifying players. This is a volume-weighted average. It gives appropriate weight to players who take many shots from a zone and appropriate weight to players who take few. Computing a simple mean of individual player PPS values would give equal weight to a player with 1 attempt and a player with 500 attempts in the same zone — that would misrepresent the true league baseline.

After computing the league average, we merge LEAGUE_AVG_PPS back into zone_metrics so that every player-zone row carries both the player's own PPS and the league average for comparison. This enriched zone_metrics table is what gets saved to disk in the next section.

The verification table shows all 11 zones sorted from highest to lowest league average PPS. The ranking should reflect known NBA efficiency patterns: the restricted area and corner threes should be near the top, mid-range zones should cluster at the bottom, and backcourt should be close to zero if any qualifying player attempted one.

In [11]:
# ── Compute league average PPS per zone ──────────────────────────────────────
# Sum FGA and POINTS across all qualifying players for each zone, then divide.
# zone_metrics already contains only the 284 qualifying players (filtered in 2.2).
league_pps = (
    zone_metrics
    .groupby("ZONE")[["FGA", "POINTS"]]
    .sum()
    .rename(columns={"FGA": "LEAGUE_FGA", "POINTS": "LEAGUE_POINTS"})
)
league_pps["LEAGUE_AVG_PPS"] = (
    league_pps["LEAGUE_POINTS"] / league_pps["LEAGUE_FGA"]
).round(4)

# Reindex to ZONE_ORDER — any zone with zero qualifying-player attempts shows NaN
league_pps = league_pps.reindex(ZONE_ORDER)

# ── Merge LEAGUE_AVG_PPS back into zone_metrics ───────────────────────────────
# Every player-zone row now carries both the player's PPS and the league baseline.
zone_metrics = zone_metrics.merge(
    league_pps["LEAGUE_AVG_PPS"].reset_index(),
    on="ZONE",
    how="left",
)

# ── Sanity check ──────────────────────────────────────────────────────────────
# Total points / total FGA across all zones should match the implied league-wide
# PPS we printed in Section 2.1 (approximately 1.091)
total_pts = league_pps["LEAGUE_POINTS"].sum()
total_fga = league_pps["LEAGUE_FGA"].sum()
overall_pps = total_pts / total_fga
print(f"League-wide overall PPS (all zones combined) : {overall_pps:.4f}")
print(f"  (this should match the implied PPS from Section 2.1 — ~1.091)\n")

# ── Verification table — sorted highest to lowest LEAGUE_AVG_PPS ──────────────
print("League average PPS per zone — highest to lowest:\n")
print(f"  {'Zone':<28} {'League FGA':>12}  {'League Pts':>12}  {'Avg PPS':>8}")
print(f"  {'─' * 65}")

for zone, row in league_pps.sort_values("LEAGUE_AVG_PPS", ascending=False).iterrows():
    if pd.isna(row["LEAGUE_AVG_PPS"]):
        print(f"  {zone:<28} {'—':>12}  {'—':>12}  {'  NaN':>8}")
    else:
        print(f"  {zone:<28} {int(row['LEAGUE_FGA']):>12,}  "
              f"{int(row['LEAGUE_POINTS']):>12,}  "
              f"{row['LEAGUE_AVG_PPS']:>8.4f}")

print(f"  {'─' * 65}")
print(f"  {'All zones combined':<28} {int(total_fga):>12,}  "
      f"{int(total_pts):>12,}  {overall_pps:>8.4f}")

# Confirm zone_metrics now has LEAGUE_AVG_PPS column
print(f"\nzone_metrics columns : {list(zone_metrics.columns)}")
print(f"zone_metrics shape   : {zone_metrics.shape[0]:,} rows x {zone_metrics.shape[1]} columns")

League-wide overall PPS (all zones combined) : 1.0928
  (this should match the implied PPS from Section 2.1 — ~1.091)

League average PPS per zone — highest to lowest:

  Zone                           League FGA    League Pts   Avg PPS
  ─────────────────────────────────────────────────────────────────
  Backcourt                              15            30    2.0000
  Restricted Area                    50,723        68,166    1.3439
  Right Corner 3                      9,015        10,623    1.1784
  Left Corner 3                       9,794        11,514    1.1756
  Above Break 3 Center               12,936        13,791    1.0661
  Above Break 3 Left                 23,621        25,170    1.0656
  Above Break 3 Right                21,070        22,038    1.0459
  Paint (Non-RA)                     38,961        35,156    0.9023
  Mid-Range Center                    4,392         3,812    0.8679
  Mid-Range Right                     7,578         6,420    0.8472
  Mid-Range Lef

## Section 2.5 — Save Output Files

We save two files here. They serve different purposes and every downstream notebook will use one or both.

`shots_cleaned.csv` is the shot-level file. It contains one row per shot attempt from the 284 qualifying players, with all the columns from the raw pull plus the five columns added in this notebook: LOC_X_FT, LOC_Y_FT, DIST_FT, IS_3PT, POINTS, and ZONE. This file is the input for Notebook 03's shot charts. No notebook after this one ever reads from data/raw/.

`zone_stats_2025_26.csv` is the player-zone summary file. It contains one row per player per zone — 2,778 rows total — with seven columns: NAME, ZONE, FGA, POINTS, PPS, SHOT_FREQ, and LEAGUE_AVG_PPS. This is the primary input for Notebooks 04, 05, and 06, which do all the tendency-vs-efficiency analysis. Because the aggregation is already done here, those notebooks never need to re-read the full shot log.

After saving, we read the file sizes from disk as a final confirmation that both writes completed successfully.

In [12]:
# ── Save 1: shots_cleaned.csv — shot-level data, qualifying players only ──────
cleaned.to_csv(SHOTS_OUT, index=False)

shots_size_mb = SHOTS_OUT.stat().st_size / 1_048_576   # bytes → MB

print(f"shots_cleaned.csv")
print(f"  Path    : {SHOTS_OUT}")
print(f"  Rows    : {len(cleaned):,}")
print(f"  Columns : {len(cleaned.columns)}")
print(f"  Size    : {shots_size_mb:.1f} MB")

# ── Save 2: zone_stats_2025_26.csv — player-zone summary ─────────────────────
# Enforce column order to match the project spec exactly
zone_cols = ["NAME", "ZONE", "FGA", "POINTS", "PPS", "SHOT_FREQ", "LEAGUE_AVG_PPS"]
zone_metrics[zone_cols].to_csv(ZONE_STATS_OUT, index=False)

zone_size_kb = ZONE_STATS_OUT.stat().st_size / 1_024   # bytes → KB

print(f"\nzone_stats_2025_26.csv")
print(f"  Path    : {ZONE_STATS_OUT}")
print(f"  Rows    : {len(zone_metrics):,}")
print(f"  Columns : {len(zone_cols)}")
print(f"  Size    : {zone_size_kb:.1f} KB")

# ── Final notebook summary ────────────────────────────────────────────────────
print(f"\n{'─' * 55}")
print(f"Notebook 02 complete.")
print(f"  Qualifying players : {cleaned['NAME'].nunique():,}")
print(f"  Shots retained     : {len(cleaned):,}")
print(f"  Player-zone rows   : {len(zone_metrics):,}")
print(f"  Zones used         : {zone_metrics['ZONE'].nunique()}")
print(f"  Season             : {SEASON}")
print(f"{'─' * 55}")

shots_cleaned.csv
  Path    : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/processed/shots_cleaned.csv
  Rows    : 185,684
  Columns : 31
  Size    : 44.9 MB

zone_stats_2025_26.csv
  Path    : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/processed/zone_stats_2025_26.csv
  Rows    : 2,778
  Columns : 7
  Size    : 155.5 KB

───────────────────────────────────────────────────────
Notebook 02 complete.
  Qualifying players : 284
  Shots retained     : 185,684
  Player-zone rows   : 2,778
  Zones used         : 11
  Season             : 2025-26
───────────────────────────────────────────────────────
